# GRPO tự viết trên Qwen2.5-0.5B-Instruct với GSM8K

Bài Basic 03 đã viết các mảnh toán của GRPO trên tensor nhỏ. Ở đây ghép chúng thành một vòng train thật: mỗi bước lấy
8 câu hỏi GSM8K, sinh 8 lời giải mỗi câu bằng chính policy, chấm bằng reward kiểm được của bài Basic 01, tính advantage
theo nhóm, log-prob của policy cũ và của reference, rồi cập nhật LoRA bằng clipped objective có KL. Không dùng `Trainer`
hay TRL: mọi dòng của vòng lặp nằm trong notebook để bạn thấy GRPO tiêu thời gian vào đâu và log những gì.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) tải model, train, đo; test bỏ qua chúng.
- Chạy thật bằng kernel `ai-gpu` (GPU 4GB là đủ). Kernel CPU chỉ chạy bản thu nhỏ để kiểm code chạy hết, rồi in lại kết
  quả GPU đã cache nếu có.
- Chấm: `uv run pytest tracks/ai-engineer/modules/15-reasoning-models/exercises/intermediate/02-grpo-from-scratch -q`

In [ ]:
# explore: tìm gốc repo, tải GSM8K nếu chưa có
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))


def dataset_dir(name):
    try:
        from data.download_dataset import ensure
    except ImportError:
        path = ROOT / "data" / "raw" / name
        if not path.is_dir():
            raise RuntimeError(f"Chưa có dữ liệu: uv run python data/download_dataset.py {name}")
        return path
    return ensure(name)


GSM = dataset_dir("gsm8k")
CACHE = GSM / "cache"
CACHE.mkdir(exist_ok=True)
print(GSM)

In [ ]:
import re
from fractions import Fraction

import torch

## 1. Dán reward (Basic 01) và objective (Basic 03)

Dán **toàn bộ** phần thân `answers.py` của bài Basic 01 (regex, `normalize_number`, `extract_answer`, ba hàm reward) và
`grpo.py` của bài Basic 03 (`group_advantages`, `kl_k3`, `clipped_surrogate`, `grpo_loss`).

In [ ]:
def normalize_number(text: str) -> str | None:
    """Dán từ bài Basic 01 (cùng mọi regex và hàm phụ nó cần)."""
    # TODO
    raise NotImplementedError


def extract_answer(completion: str) -> str | None:
    """Dán từ bài Basic 01."""
    # TODO
    raise NotImplementedError


def format_reward(completion: str) -> float:
    """Dán từ bài Basic 01."""
    # TODO
    raise NotImplementedError


def correctness_reward(completion: str, gold: str) -> float:
    """Dán từ bài Basic 01."""
    # TODO
    raise NotImplementedError


def total_reward(completion: str, gold: str, format_weight: float = 0.2) -> float:
    """Dán từ bài Basic 01."""
    # TODO
    raise NotImplementedError

In [ ]:
def group_advantages(rewards: torch.Tensor, group_size: int, scale: bool = True, eps: float = 1e-4) -> torch.Tensor:
    """Dán từ bài Basic 03."""
    # TODO
    raise NotImplementedError


def kl_k3(logp: torch.Tensor, ref_logp: torch.Tensor) -> torch.Tensor:
    """Dán từ bài Basic 03."""
    # TODO
    raise NotImplementedError


def clipped_surrogate(logp: torch.Tensor, old_logp: torch.Tensor, advantages: torch.Tensor,
                      eps_low: float = 0.2, eps_high: float | None = None) -> torch.Tensor:
    """Dán từ bài Basic 03."""
    # TODO
    raise NotImplementedError


def grpo_loss(logp, old_logp, ref_logp, advantages, mask, beta=0.04, eps_low=0.2, eps_high=None,
              aggregation="sequence"):
    """Dán từ bài Basic 03."""
    # TODO
    raise NotImplementedError

## 2. Mask và log-prob của phần completion

- `completion_mask(completion_ids, stop_ids) -> Tensor`: cùng shape `(B, C)`, 1 từ đầu tới **và gồm** token đầu tiên
  thuộc `stop_ids` (model phải học dừng), 0 sau đó; hàng không có token dừng (bị cắt ở `max_new_tokens`) thì 1 cả hàng.
- `completion_logprobs(model, prompt_ids, prompt_mask, completion_ids, completion_mask, temperature=1.0) -> Tensor (B, C)`:
  log-prob của từng token completion dưới `model`. Prompt đã **pad trái** (`prompt_mask` 0 ở chỗ pad), completion pad
  phải. Ghép prompt và completion, gọi `model(input_ids=..., attention_mask=..., logits_to_keep=C + 1)` **một lần** (chỉ
  lấy logits của C + 1 vị trí cuối: với vocab 151 936 token, logits cả chuỗi chiếm phần lớn bộ nhớ), chia logits cho
  `temperature`, log-softmax, lấy log-prob của token completion (token ở vị trí `t` được dự đoán từ logits ở vị trí
  `t - 1`). Vị trí có `completion_mask = 0` trả 0. **Không** bọc `no_grad`: loss cần gradient qua đây.

In [ ]:
def completion_mask(completion_ids: torch.Tensor, stop_ids: set[int]) -> torch.Tensor:
    """1 tới và gồm token dừng đầu tiên, 0 sau đó; không có token dừng thì 1 cả hàng."""
    # TODO
    raise NotImplementedError


def completion_logprobs(model, prompt_ids: torch.Tensor, prompt_mask: torch.Tensor, completion_ids: torch.Tensor,
                        completion_mask: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
    """Log-prob (B, C) của token completion dưới model; 0 ngoài completion_mask; giữ gradient."""
    # TODO
    raise NotImplementedError

## 3. Model, LoRA và cách đo

`Qwen2.5-0.5B-Instruct`, LoRA r = 16 trên mọi lớp tuyến tính (module 14 mục 3). **Reference** không cần một bản model
thứ hai: tắt adapter (`with model.disable_adapter():`) là ra đúng model gốc. Prompt là CoT của bài Intermediate 01
(`\boxed{}`), câu train lấy từ tập train GSM8K (trộn theo seed 0), câu đo là 200 câu đầu tập test, greedy.

In [ ]:
# explore: dữ liệu, model, LoRA, hàm sinh và hàm đo
import json
import math
import time

import numpy as np
import pandas as pd
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
SYSTEM_COT = "Please reason step by step, and put your final answer within \\boxed{}."
CFG = dict(steps=30, prompts=8, group=8, max_new=320, temperature=0.7, lr=2e-5, beta=0.04, iters=2, micro=2,
           n_eval=200, eval_max_new=512) if GPU else \
      dict(steps=2, prompts=2, group=4, max_new=48, temperature=0.7, lr=2e-5, beta=0.04, iters=2, micro=4,
           n_eval=6, eval_max_new=64)
print(CFG)

train = pd.read_parquet(GSM / "main" / "train-00000-of-00001.parquet").sample(frac=1.0, random_state=0)
test = pd.read_parquet(GSM / "main" / "test-00000-of-00001.parquet").iloc[:CFG["n_eval"]]
gold = lambda a: normalize_number(a.split("####")[-1])

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
STOP = {tok.eos_token_id, tok.pad_token_id, tok.convert_tokens_to_ids("<|im_end|>")}
torch.manual_seed(0)
base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16 if GPU else torch.float32).to(DEVICE)
base.gradient_checkpointing_enable()
base.enable_input_require_grads()
model = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear",
                                        task_type="CAUSAL_LM"))


def encode_prompts(questions):
    texts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM_COT}, {"role": "user", "content": q}],
                                     tokenize=False, add_generation_prompt=True) for q in questions]
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False)
    return enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE)


@torch.no_grad()
def generate(prompt_ids, prompt_mask, max_new, temperature):
    model.eval()
    kw = (dict(do_sample=True, temperature=temperature, top_p=1.0, top_k=0) if temperature > 0
          else dict(do_sample=False, temperature=None, top_p=None, top_k=None))
    out = model.generate(input_ids=prompt_ids, attention_mask=prompt_mask, max_new_tokens=max_new,
                         pad_token_id=tok.pad_token_id, **kw)
    completion = out[:, prompt_ids.shape[1]:]
    mask = completion_mask(completion, STOP)
    texts = tok.batch_decode(completion * mask + tok.pad_token_id * (1 - mask), skip_special_tokens=True)
    return completion, mask, texts


def evaluate(name):
    t0 = time.perf_counter()
    texts, lengths = [], []
    for s in range(0, len(test), 64):
        ids, mask = encode_prompts(list(test.question[s:s + 64]))
        _, cmask, out = generate(ids, mask, CFG["eval_max_new"], 0.0)
        texts += out
        lengths += cmask.sum(1).tolist()
    golds = [gold(a) for a in test.answer]
    res = {"accuracy": float(np.mean([extract_answer(t) == g for t, g in zip(texts, golds)])),
           "boxed_correct": float(np.mean([correctness_reward(t, g) for t, g in zip(texts, golds)])),
           "format": float(np.mean([format_reward(t) for t in texts])),
           "mean_tokens": float(np.mean(lengths)), "seconds": round(time.perf_counter() - t0, 1),
           "correct": [bool(extract_answer(t) == g) for t, g in zip(texts, golds)], "examples": texts[:3]}
    print(name, {k: v for k, v in res.items() if k not in ("correct", "examples")})
    return res


results = {"config": CFG, "before": evaluate("trước GRPO")}

## 4. Vòng GRPO

Mỗi bước:

1. Lấy `prompts` câu train, lặp mỗi câu `group` lần, sinh bằng policy hiện tại (`temperature` 0.7).
2. Reward `total_reward` (đúng đáp số trong `\boxed{}` + 0.2 × đúng khung), advantage theo nhóm.
3. `old_logp` (policy lúc sinh) và `ref_logp` (adapter tắt), cả hai không gradient, cùng `temperature` với lúc sinh.
4. `iters` lượt gradient trên cùng batch (lượt đầu ratio bằng 1; lượt sau mới có phép cắt), chia micro-batch để vừa
   GPU 4GB, `grpo_loss` gộp theo chuỗi, clip norm 1.0, AdamW.

Log mỗi bước: reward, tỉ lệ đúng, tỉ lệ đúng khung, độ dài completion, tỉ lệ nhóm std = 0, KL, clip fraction, thời gian
sinh và thời gian cập nhật.

In [ ]:
# explore: train
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG["lr"], weight_decay=0.0)
G, T = CFG["group"], CFG["temperature"]
history = []
t_start = time.perf_counter()
if GPU:
    torch.cuda.reset_peak_memory_stats()
for step in range(CFG["steps"]):
    rows = train.iloc[step * CFG["prompts"]:(step + 1) * CFG["prompts"]]
    questions = [q for q in rows.question for _ in range(G)]
    golds = [gold(a) for a in rows.answer for _ in range(G)]
    prompt_ids, prompt_mask = encode_prompts(questions)
    t0 = time.perf_counter()
    completion, cmask, texts = generate(prompt_ids, prompt_mask, CFG["max_new"], T)
    t_gen = time.perf_counter() - t0
    rewards = torch.tensor([total_reward(t, g) for t, g in zip(texts, golds)], device=DEVICE)
    adv = group_advantages(rewards, G)
    zero_std = float((rewards.view(-1, G).std(dim=1) == 0).float().mean())

    chunks = [slice(s, s + CFG["micro"]) for s in range(0, len(texts), CFG["micro"])]
    with torch.no_grad():
        old_logp = torch.cat([completion_logprobs(model, prompt_ids[c], prompt_mask[c], completion[c], cmask[c], T)
                              for c in chunks])
        with model.disable_adapter():
            ref_logp = torch.cat([completion_logprobs(model, prompt_ids[c], prompt_mask[c], completion[c], cmask[c], T)
                                  for c in chunks])
    model.train()
    stats = []
    for _ in range(CFG["iters"]):
        opt.zero_grad(set_to_none=True)
        for c in chunks:
            logp = completion_logprobs(model, prompt_ids[c], prompt_mask[c], completion[c], cmask[c], T)
            loss, st = grpo_loss(logp, old_logp[c], ref_logp[c], adv[c], cmask[c], beta=CFG["beta"])
            (loss * len(range(*c.indices(len(texts)))) / len(texts)).backward()
            stats.append(st)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        opt.step()
    history.append({"step": step, "reward": float(rewards.mean()),
                    "correct": float(np.mean([correctness_reward(t, g) for t, g in zip(texts, golds)])),
                    "format": float(np.mean([format_reward(t) for t in texts])),
                    "length": float(cmask.sum(1).float().mean()), "zero_std": zero_std,
                    "kl": float(np.mean([s["kl"] for s in stats])),
                    "clip_frac": float(np.mean([s["clip_frac"] for s in stats])),
                    "t_gen": round(t_gen, 1), "t_step": round(time.perf_counter() - t0, 1)})
    h = history[-1]
    print(f"bước {step:3d} reward {h['reward']:.3f} đúng {h['correct']:.2f} khung {h['format']:.2f} "
          f"dài {h['length']:.0f} std0 {h['zero_std']:.2f} kl {h['kl']:.4f} clip {h['clip_frac']:.3f} "
          f"sinh {h['t_gen']}s / bước {h['t_step']}s")
results["history"] = history
results["train_minutes"] = round((time.perf_counter() - t_start) / 60, 1)
results["peak_gib"] = round(torch.cuda.max_memory_allocated() / 2**30, 2) if GPU else None
results["after"] = evaluate("sau GRPO")
path = CACHE / ("m15-grpo-scratch.json" if GPU else "m15-grpo-scratch-cpu.json")
path.write_text(json.dumps(results, ensure_ascii=False, indent=1), encoding="utf-8")
print("đã ghi", path, "| train", results["train_minutes"], "phút | VRAM đỉnh", results["peak_gib"], "GiB")

In [ ]:
# explore: đường cong và bảng trước/sau (kernel CPU in thêm kết quả GPU đã cache nếu có)
import matplotlib.pyplot as plt


def report(res, title):
    h = pd.DataFrame(res["history"])
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
    for ax, cols in zip(axes, (["reward", "correct", "format"], ["length"], ["zero_std"], ["kl", "clip_frac"])):
        for col in cols:
            ax.plot(h["step"], h[col].rolling(5, min_periods=1).mean(), label=col)
        ax.set_xlabel("bước")
        ax.legend()
    fig.suptitle(title)
    plt.show()
    table = pd.DataFrame({k: {m: res[k][m] for m in ("accuracy", "boxed_correct", "format", "mean_tokens")}
                          for k in ("before", "after")}).T
    display(table.round(3))
    a, b = np.array(res["before"]["correct"]), np.array(res["after"]["correct"])
    print(f"câu đổi đúng -> sai: {int((a & ~b).sum())}, sai -> đúng: {int((~a & b).sum())} / {len(a)}")
    print("5 bước đầu:", h.head(5)[["reward", "correct", "format", "zero_std"]].mean().round(3).to_dict())
    print("5 bước cuối:", h.tail(5)[["reward", "correct", "format", "zero_std"]].mean().round(3).to_dict())


report(results, "lần chạy này")
gpu_cache = CACHE / "m15-grpo-scratch.json"
if not GPU and gpu_cache.exists():
    report(json.loads(gpu_cache.read_text(encoding="utf-8")), "kết quả GPU đã cache")

## Đọc kết quả

- Thời gian mỗi bước chia cho sinh và cập nhật thế nào? Vì sao các thư viện GRPO tối ưu phần sinh trước (vLLM, sinh
  song song)?
- Tỉ lệ nhóm std = 0 bao nhiêu? Mỗi nhóm như vậy là 8 lời giải sinh ra mà không đóng góp gradient.
- Reward tăng đến từ phần nào: đúng đáp số hay đúng khung? So `accuracy` (bộ trích dễ dãi) với `boxed_correct` (đáp
  số trong hộp, như reward) trước và sau: hai cột kể cùng một câu chuyện không?
- Chênh lệch trước/sau trên 200 câu có vượt nhiễu không (đếm số câu đổi đúng sai theo hai chiều)? 30 bước với 8 câu mỗi
  bước là 240 câu train: bạn kỳ vọng gì?
- KL và clip fraction ở mức nào so với vùng bình thường của PPO (module 07 mục 6)?

Sau đó làm phần trắc nghiệm của bài trong app.